# Estatísticas de Variáveis Numéricas

---

## Objetivo

Calcular **média, mediana, mínimo, máximo, amplitude, desvio padrão e coeficiente de variação (CV)** para as principais variáveis numéricas, além de analisar a distribuição temporal dos registros.

### Métricas calculadas

| Métrica | Fórmula | Interpretação |
|:---|:---|:---|
| Média (μ) | Σxᵢ / n | Centro de gravidade da distribuição; sensível a outliers |
| Mediana | Valor central ordenado | Resistente a outliers; melhor medida de tendência central quando a distribuição é assimétrica |
| Amplitude | Máx − Mín | Extensão total da variação |
| Desvio Padrão (σ) | √(Σ(xᵢ−μ)² / n) | Dispersão média em torno da média |
| CV | (σ / μ) × 100 | Dispersão relativa — permite comparar variáveis em escalas diferentes. CV > 30% indica alta variabilidade |


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})

AZUL_ESCURO = '#1a3a5c'
AZUL_MEDIO  = '#2d6a9f'
AZUL_CLARO  = '#4a90c4'
LARANJA     = '#e07b39'

BASE_PATH = '../data/raw/'
pesquisas    = pd.read_csv(BASE_PATH + 'pesquisas.csv',    sep=';')
empresas     = pd.read_csv(BASE_PATH + 'empresas.csv',     sep=';')
respondentes = pd.read_csv(BASE_PATH + 'respondentes.csv', sep=';')
questoes     = pd.read_csv(BASE_PATH + 'questoes.csv',     sep=';')

def estatisticas_numericas(series, nome, unidade=''):
    """Calcula e retorna dict com estatísticas descritivas completas."""
    s = series.dropna()
    media   = s.mean()
    mediana = s.median()
    minimo  = s.min()
    maximo  = s.max()
    amp     = maximo - minimo
    std     = s.std()
    cv      = (std / media * 100) if media != 0 else float('nan')
    q1      = s.quantile(0.25)
    q3      = s.quantile(0.75)
    iqr     = q3 - q1
    skew    = s.skew()
    return {
        'Variável': nome,
        'n válido': len(s),
        'n nulo': series.isnull().sum(),
        'Média': round(media, 2),
        'Mediana': round(mediana, 2),
        'Mín': round(minimo, 2),
        'Máx': round(maximo, 2),
        'Amplitude': round(amp, 2),
        'Desv. Padrão': round(std, 2),
        'CV (%)': round(cv, 1),
        'Q1': round(q1, 2),
        'Q3': round(q3, 2),
        'IQR': round(iqr, 2),
        'Assimetria': round(skew, 2),
        'Unidade': unidade,
    }

print('Setup concluído.')

---

## 1. Variáveis numéricas de `pesquisas.csv`

Estas são as variáveis mais críticas para os KPIs operacionais de tempo de ciclo e participação.


In [ ]:
# Preparação: converter part_mercado para numérico
pesquisas['part_mercado_num'] = (
    pesquisas['part_mercado']
    .str.replace('%', '', regex=False)
    .str.replace(',', '.', regex=False)
    .astype(float, errors='ignore')
)
# tenta conversão mais segura
pesquisas['part_mercado_num'] = pd.to_numeric(
    pesquisas['part_mercado'].str.replace('%','').str.replace(',','.'), errors='coerce')

variaveis_pesquisas = [
    (pesquisas['participantes'],                                     'participantes',                       'empresas'),
    (pesquisas['part_mercado_num'],                                  'part_mercado (% mercado)',             '%'),
    (pesquisas['dias_uteis_para_entrega'],                           'dias_uteis_para_entrega',             'dias úteis'),
    (pesquisas['dias entre a solicitação e a divulgação'],           'dias solicitação→divulgação',         'dias úteis'),
    (pesquisas['dias para tabulação'],                               'dias para tabulação',                 'dias úteis'),
    (pesquisas['dias úteis aberta'],                                 'dias úteis aberta',                   'dias úteis'),
    (pesquisas['tempo_preenchimento'],                               'tempo_preenchimento',                 'minutos'),
]

stats_pesquisas = [estatisticas_numericas(s, n, u) for s, n, u in variaveis_pesquisas]
df_stats = pd.DataFrame(stats_pesquisas)

print('=== Estatísticas descritivas — variáveis numéricas de pesquisas.csv ===')
cols_show = ['Variável', 'n válido', 'Média', 'Mediana', 'Mín', 'Máx', 'Amplitude', 'Desv. Padrão', 'CV (%)', 'Unidade']
print(df_stats[cols_show].to_string(index=False))

In [ ]:
# Boxplots das variáveis de tempo
variaveis_tempo = [
    (pesquisas['dias entre a solicitação e a divulgação'].dropna(), 'Solicitação→Divulgação'),
    (pesquisas['dias para tabulação'].dropna(),                     'Tabulação'),
    (pesquisas['dias_uteis_para_entrega'].dropna(),                 'Ciclo total'),
    (pesquisas['dias úteis aberta'].dropna(),                       'Coleta aberta'),
]

fig, ax = plt.subplots(figsize=(10, 5))
data_box = [s.values for s, _ in variaveis_tempo]
labels    = [n for _, n in variaveis_tempo]

bp = ax.boxplot(data_box, vert=False, patch_artist=True, labels=labels,
                medianprops={'color': LARANJA, 'linewidth': 2},
                flierprops={'marker': 'o', 'markersize': 4, 'alpha': 0.5, 'markeredgecolor': AZUL_ESCURO})

for patch in bp['boxes']:
    patch.set_facecolor(AZUL_CLARO)
    patch.set_alpha(0.7)

# Adicionar média como triângulo
for i, (s, _) in enumerate(variaveis_tempo, 1):
    ax.plot(s.mean(), i, marker='^', color=AZUL_ESCURO, markersize=7, zorder=3, label='Média' if i == 1 else '')

ax.legend(loc='lower right')
ax.set_xlabel('Dias úteis')
ax.set_title('Distribuição das variáveis de tempo (dias úteis)\nLaranja = mediana  |  Triângulo = média', fontweight='bold')
plt.tight_layout()
plt.savefig('fig_11_boxplot_tempos.png', bbox_inches='tight')
plt.show()

# Interpretar
print('\nInterpretação dos boxplots:')
for s, nome in variaveis_tempo:
    skew = s.skew()
    direcao = 'assimetria positiva (cauda direita)' if skew > 0.5 else 'assimetria negativa (cauda esquerda)' if skew < -0.5 else 'aproximadamente simétrica'
    print(f'  {nome:30s} | média={s.mean():.1f}d | mediana={s.median():.1f}d | assimetria: {direcao}')

---

## 2. Histogramas: distribuição de `participantes` e `dias_uteis_para_entrega`


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, col, titulo, unidade, cor in [
    (axes[0], 'participantes', 'Participantes por pesquisa', 'empresas', AZUL_CLARO),
    (axes[1], 'dias_uteis_para_entrega', 'Dias úteis para entrega', 'dias úteis', AZUL_MEDIO),
]:
    s = pesquisas[col].dropna()
    ax.hist(s, bins=20, color=cor, edgecolor='white', alpha=0.85)
    ax.axvline(s.mean(),   color=LARANJA,     linewidth=1.8, linestyle='--', label=f'Média: {s.mean():.1f}')
    ax.axvline(s.median(), color=AZUL_ESCURO, linewidth=1.8, linestyle='-',  label=f'Mediana: {s.median():.1f}')
    ax.set_xlabel(unidade)
    ax.set_ylabel('Frequência')
    ax.set_title(titulo, fontweight='bold')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('fig_12_histogramas_numericas.png', bbox_inches='tight')
plt.show()

# Comentário sobre o CV
for col, nome in [('participantes', 'Participantes'), ('dias_uteis_para_entrega', 'Dias úteis entrega')]:
    s = pesquisas[col].dropna()
    cv = s.std() / s.mean() * 100
    alerta = '⚠ Alta variabilidade' if cv > 30 else '✓ Variabilidade moderada'
    print(f'{nome:30s} | CV = {cv:.1f}% | {alerta}')

> **CV alto** (>30%) indica que a média isolada é um mau representante — use mediana e IQR para comunicar esses indicadores nos dashboards.

---

## 3. `empresas.% de mercado` — Concentração de mercado


In [ ]:
empresas['pct_num'] = pd.to_numeric(
    empresas['% de mercado'].str.replace('%','').str.replace(',','.'), errors='coerce')

s_mercado = empresas['pct_num']
st_mercado = estatisticas_numericas(s_mercado, '% de mercado (empresas)', '%')

print('=== empresas.% de mercado ===')
for k, v in st_mercado.items():
    print(f'  {k:20s}: {v}')

print()
print('Concentração: top 10 empresas por market share')
top10 = empresas[['Nome Fantasia', 'pct_num']].nlargest(10, 'pct_num')
print(top10.to_string(index=False))
print(f'\nTop 10 concentram: {top10["pct_num"].sum():.2f}% do mercado')

# Gráfico de concentração (Pareto-style)
emp_sorted = empresas.nlargest(50, 'pct_num').copy()
emp_sorted['acumulado'] = emp_sorted['pct_num'].cumsum()

fig, ax1 = plt.subplots(figsize=(11, 4))
ax1.bar(range(len(emp_sorted)), emp_sorted['pct_num'], color=AZUL_CLARO, alpha=0.8)
ax1.set_xlabel('Empresas (ordenadas por market share)')
ax1.set_ylabel('Market share individual (%)', color=AZUL_CLARO)

ax2 = ax1.twinx()
ax2.plot(range(len(emp_sorted)), emp_sorted['acumulado'], color=LARANJA, linewidth=2)
ax2.set_ylabel('Market share acumulado (%)', color=LARANJA)
ax2.axhline(80, color='red', linewidth=0.8, linestyle='--', alpha=0.5, label='80%')

ax1.set_title('Concentração de market share — top 50 empresas', fontweight='bold')
ax2.legend(loc='center right')
plt.tight_layout()
plt.savefig('fig_13_market_share.png', bbox_inches='tight')
plt.show()

---

## 4. `questoes.qtd_alternativas` — Complexidade das perguntas


In [ ]:
st_alt = estatisticas_numericas(questoes['qtd_alternativas'], 'qtd_alternativas', 'alternativas')

print('=== questoes.qtd_alternativas ===')
cols_show = ['n válido', 'n nulo', 'Média', 'Mediana', 'Mín', 'Máx', 'Amplitude', 'Desv. Padrão', 'CV (%)']
for c in cols_show:
    print(f'  {c:20s}: {st_alt[c]}')

# Por tipo de questão
print('\nMédia de alternativas por tipo de questão:')
print(questoes.groupby('tipo')['qtd_alternativas'].agg(['mean','median','count']).round(1))

---

## 5. Distribuição temporal — pesquisas por ano e mês


In [ ]:
pesquisas['data_sol_parsed'] = pd.to_datetime(
    pesquisas['data_solicitacao'], dayfirst=True, errors='coerce')
pesquisas['ano'] = pesquisas['data_sol_parsed'].dt.year
pesquisas['mes'] = pesquisas['data_sol_parsed'].dt.month

por_ano = pesquisas.groupby('ano').size().reset_index(name='pesquisas')
# Filtrar apenas anos completos (2023-2025)
por_ano_completo = por_ano[por_ano['ano'].between(2023, 2025)]

print('=== Distribuição temporal de pesquisas ===')
print('Por ano (data_solicitacao):')
print(por_ano.to_string(index=False))
print()
print('Nota: 2022 possui apenas 5 registros (base parcial); 2026 e 2027 estão incompletos.')
print(f'Crescimento 2023→2025: {((119-95)/95*100):.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Gráfico 1: por ano
ax = axes[0]
cores_ano = [AZUL_ESCURO if a in [2023, 2024, 2025] else AZUL_CLARO for a in por_ano['ano']]
bars = ax.bar(por_ano['ano'].astype(str), por_ano['pesquisas'], color=cores_ano, width=0.6)
for b, v in zip(bars, por_ano['pesquisas']):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.5, str(v), ha='center', fontsize=9)
ax.set_xlabel('Ano')
ax.set_ylabel('Pesquisas')
ax.set_title('Pesquisas solicitadas por ano', fontweight='bold')
ax.set_ylim(0, por_ano['pesquisas'].max() * 1.2)

# Gráfico 2: por mês (anos 2023-2025 completos)
ax2 = axes[1]
por_mes = pesquisas[pesquisas['ano'].between(2023, 2025)].groupby('mes').size().reset_index(name='pesquisas')
meses = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']
ax2.bar(por_mes['mes'], por_mes['pesquisas'], color=AZUL_MEDIO, width=0.7)
ax2.set_xticks(range(1, 13))
ax2.set_xticklabels(meses, rotation=30, ha='right')
ax2.set_ylabel('Pesquisas')
ax2.set_title('Pesquisas por mês (2023–2025)', fontweight='bold')
for i, row in por_mes.iterrows():
    ax2.text(row['mes'], row['pesquisas'] + 0.3, str(row['pesquisas']), ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('fig_14_distribuicao_temporal.png', bbox_inches='tight')
plt.show()

In [ ]:
# Distribuição temporal: volume por área ao longo dos anos
pesquisas_com_ano = pesquisas[pesquisas['ano'].between(2023, 2025)].copy()
top_areas = pesquisas['area'].value_counts().head(5).index.tolist()

vol_area_ano = (
    pesquisas_com_ano[pesquisas_com_ano['area'].isin(top_areas)]
    .groupby(['ano', 'area'])
    .size()
    .unstack(fill_value=0)
)

print('=== Volume de pesquisas por área × ano (top 5 áreas, 2023-2025) ===')
print(vol_area_ano)

fig, ax = plt.subplots(figsize=(10, 4.5))
cores_areas = [AZUL_ESCURO, AZUL_MEDIO, AZUL_CLARO, LARANJA, '#27ae60']
vol_area_ano.plot(kind='bar', ax=ax, color=cores_areas, width=0.7, edgecolor='white')
ax.set_xlabel('Ano')
ax.set_ylabel('Pesquisas')
ax.set_title('Volume de pesquisas por área × ano (top 5 áreas)', fontweight='bold')
ax.legend(title='Área', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.savefig('fig_15_vol_area_ano.png', bbox_inches='tight')
plt.show()

---

## 6. Tabela-resumo consolidada


In [ ]:
# Calcular também para empresas.pct e questoes.qtd_alternativas
empresas['pct_num'] = pd.to_numeric(
    empresas['% de mercado'].str.replace('%','').str.replace(',','.'), errors='coerce')

todas_stats = stats_pesquisas + [
    estatisticas_numericas(empresas['pct_num'],          '% mercado (empresas)', '%'),
    estatisticas_numericas(questoes['qtd_alternativas'], 'qtd_alternativas',     'alternativas'),
]

df_todas = pd.DataFrame(todas_stats)
cols_resumo = ['Variável', 'n válido', 'Média', 'Mediana', 'Mín', 'Máx', 'Desv. Padrão', 'CV (%)', 'Assimetria', 'Unidade']

print('=== TABELA-RESUMO: Estatísticas das variáveis numéricas ===')
print(df_todas[cols_resumo].to_string(index=False))
print()
print('Classificação do CV:')
for _, row in df_todas.iterrows():
    cv = row['CV (%)']
    nivel = 'Baixa (<15%)' if cv < 15 else 'Moderada (15-30%)' if cv < 30 else '⚠ Alta (>30%)'
    print(f'  {row["Variável"]:45s} CV={cv:6.1f}% | Variabilidade: {nivel}')

## Conclusões

1. **Dias úteis para entrega:** mediana de 19 dias úteis, mas média de 21,1 — a distribuição tem assimetria positiva com alguns outliers acima de 60 dias. **Para KPIs de SLA, usar mediana é mais representativo.**

2. **Participantes:** media de 60,7 empresas por pesquisa, com CV de 68% — altíssima variabilidade. Pesquisas de área de RH tendem a ter mais participantes que pesquisas regulatórias específicas.

3. **Market share:** mediana zero (a maioria das empresas tem share insignificante ou zero); a média é puxada pelas poucas empresas com alta participação. **Análise de representatividade deve usar soma, não média.**

4. **Tendência temporal:** crescimento consistente de pesquisas de 2023 (95) para 2025 (119), +25,3%. O pipeline está crescendo — argumento para automação urgente.

5. **Sazonalidade:** a distribuição por mês revela possíveis picos — a confirmar com anos individuais para evitar confundir variação inter-anual com sazonalidade.
